# Notebook 05 — Statistics & Random Numbers

**numpy-mastery** · Module 05 of 06

> **Goal**: compute descriptive statistics correctly, understand the modern
> `np.random.default_rng` API, and generate any distribution you need for
> data simulation and ML experiments.

**What you'll be able to do after this notebook:**
- Compute mean, std, percentiles, correlation on any axis
- Understand the difference between population and sample statistics
- Use the modern Generator API (never `np.random.seed` again)
- Generate data from uniform, normal, integer, and other distributions
- Shuffle and sample arrays without replacement

---

## 0 · Setup

In [3]:
import numpy as np

---
## 1 · Descriptive statistics

These functions work on any array and accept an `axis` argument just like
the aggregations in Module 03.

In [4]:
data = np.array([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0])

print(f"mean : {np.mean(data):.4f}")    # 5.0
print(f"median : {np.median(data):.4f}")  # 4.5  (middle value)
print(f"min : {data.min()}")
print(f"max : {data.max()}")
print(f"range : {data.max() - data.min()}")
print(f"sum : {data.sum()}")

mean : 5.0000
median : 4.5000
min : 2.0
max : 9.0
range : 7.0
sum : 40.0


---
## 2 · Standard deviation and variance — population vs sample

This is a critical distinction that causes real bugs if ignored.

| Parameter | Formula | When to use |
|-----------|---------|-------------|
| `ddof=0` (default) | divide by N | you have the **entire population** |
| `ddof=1` | divide by N-1 | you have a **sample** from a larger population |

In almost all real data science work you have a sample — use `ddof=1`.


In [5]:
data = np.array([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0])

pop_std = np.std(data, ddof=0)   # population std (default)
samp_std = np.std(data, ddof=1)   # sample std (Bessel's correction)


print(f"population std  (ddof=0): {pop_std:.4f}")
print(f"sample std (ddof=1): {samp_std:.4f}")
print(f"population var (ddof=0): {np.var(data, ddof=0):.4f}")
print(f"sample var (ddof=1): {np.var(data, ddof=1):.4f}")

population std  (ddof=0): 2.0000
sample std (ddof=1): 2.1381
population var (ddof=0): 4.0000
sample var (ddof=1): 4.5714


---
## 3 · Percentiles and quantiles

A percentile answers: "what value is this fraction of the data below?"  
The 50th percentile is the median.

In [6]:
data = np.array([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0])

q25 = np.percentile(data, 25)    # Q1 — 25th percentile
q50 = np.percentile(data, 50)    # Q2 — median
q75 = np.percentile(data, 75)    # Q3 — 75th percentile
iqr = q75 - q25                  # Interquartile range

print(f"Q1 = {q25}")
print(f"Q2 = {q50}")
print(f"Q3 = {q75}")
print(f"IQR = {iqr}")

# Request multiple at once
q = np.percentile(data, [0, 25, 50, 75, 100])
print("five-number summary:", q)

# np.quantile is the same but takes fractions in [0, 1]
print("90th percentile:", np.quantile(data, 0.9))


Q1 = 4.0
Q2 = 4.5
Q3 = 5.5
IQR = 1.5
five-number summary: [2.  4.  4.5 5.5 9. ]
90th percentile: 7.6


---
## 4 · Statistics on 2-D arrays with `axis`

The same `axis` logic from Module 03 applies to all stat functions.

In [9]:
X = np.array([[10, 20, 30],
              [ 5, 15, 25],
              [40, 10, 20]])  # shape (3, 3) 

print("column means (axis=0):", X.mean(axis=0))  # per feature
print("row means (axis=1):", X.mean(axis=1))  # per sample

print("column stds  (axis=0):", X.std(axis=0, ddof=1))
print("column medians :", np.median(X, axis=0))

column means (axis=0): [18.33333333 15.         25.        ]
row means (axis=1): [20.         15.         23.33333333]
column stds  (axis=0): [18.92969449  5.          5.        ]
column medians : [10. 15. 25.]


---
## 5 · Correlation and covariance

**Covariance** measures how two variables change together.  
**Correlation** (Pearson r) is covariance normalised to `[-1, 1]` — scale-free.

| r value | Interpretation |
|---------|---------------|
| +1 | perfect positive linear relationship |
| 0 | no linear relationship |
| -1 | perfect negative linear relationship |

In [10]:
x = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y = np.array([2.1, 3.9, 6.2, 7.8, 10.1])   # roughly y ≈ 2x
z = np.array([9.9, 8.1, 6.0, 3.9, 2.0])   # roughly z ≈ -2x + 12

# corrcoef returns a correlation MATRIX — diagonal is always 1.0
corr_xy = np.corrcoef(x, y)
print("corr(x,y) matrix:\n", corr_xy)
print("r =", corr_xy[0, 1])   # off-diagonal element is the actual r

corr_xz = np.corrcoef(x, z)
print("\nr(x,z) =", corr_xz[0, 1])   # should be close to -1

# covariance matrix
print("\ncov matrix x,y:\n", np.cov(x, y))

corr(x,y) matrix:
 [[1.         0.99865176]
 [0.99865176 1.        ]]
r = 0.9986517555689656

r(x,z) = -0.999650183642878

cov matrix x,y:
 [[2.5   4.975]
 [4.975 9.927]]


---
## 6 · The old API — what to avoid

You will see this style everywhere online. It works but has problems:

In [11]:
# OLD STYLE — do not use in new code
np.random.seed(42)          # global state — causes hidden bugs in parallel code
arr = np.random.randn(5)    # samples from standard normal
arr = np.random.rand(5)     # uniform [0,1)
arr = np.random.randint(0, 10, size=5)

# The problem: any function anywhere that calls np.random changes the global
# state, making results non-reproducible in complex code.
print("old style works but is fragile:", arr)

old style works but is fragile: [1 7 5 1 4]


---
## 7 · The modern API — `np.random.default_rng`

Since NumPy 1.17 (2019). Always use this in new code.  
The key idea: the random state lives in a **local object**, not globally.


In [12]:
# Create a Generator — seed makes it reproducible
rng = np.random.default_rng(seed=42)

# Every call uses THIS rng object — no global state pollution
arr = rng.random(5)
print("uniform [0,1]:", arr)

# Run again — you get the SAME numbers (reproducible)
rng2 = np.random.default_rng(seed=42)
arr2 = rng2.random(5)
print("same seed :", arr2)
print("identical :", np.array_equal(arr, arr2))

# Different seed → different numbers
rng3 = np.random.default_rng(seed=99)
arr3 = rng3.random(5)
print("diff seed :", arr3)

uniform [0,1]: [0.77395605 0.43887844 0.85859792 0.69736803 0.09417735]
same seed : [0.77395605 0.43887844 0.85859792 0.69736803 0.09417735]
identical : True
diff seed : [0.50603067 0.56509163 0.51191596 0.97218637 0.61490314]


---
## 8 · Generating data from distributions

In [13]:
rng = np.random.default_rng(42)

# Uniform floats in [0, 1)
print(rng.random(5))

# Uniform floats in [low, high)
print(rng.uniform(low=-1.0, high=1.0, size=5))

# Uniform integers in [low, high)   — high is EXCLUSIVE
print(rng.integers(low=0, high=10, size=5))

# Normal (Gaussian): mean=mu, std=sigma
print(rng.normal(loc=0.0, scale=1.0, size=5))     # standard normal
print(rng.normal(loc=5.0, scale=2.0, size=(3,4))) # 3×4 matrix


[0.77395605 0.43887844 0.85859792 0.69736803 0.09417735]
[ 0.9512447   0.5222794   0.57212861 -0.74377273 -0.09922812]
[5 3 1 9 7]
[ 1.12724121  0.46750934 -0.85929246  0.36875078 -0.9588826 ]
[[6.7569006  4.90014818 4.63027527 3.63814091]
 [7.44508268 4.69094104 4.14334436 4.2957329 ]
 [6.06461837 5.73088813 5.82546522 5.86164201]]


In [14]:
rng = np.random.default_rng(42)

# Other useful distributions
print("binomial :", rng.binomial(n=10, p=0.3, size=5))      # coin flips
print("poisson :", rng.poisson(lam=3.0, size=5))            # event counts
print("exponential:", rng.exponential(scale=2.0, size=5))   # waiting times
print("beta :", rng.beta(a=2.0, b=5.0, size=5))             # probabilities

binomial : [4 3 5 4 1]
poisson : [4 5 1 7 1]
exponential: [0.35926451 1.3706409  0.77736047 2.52841372 1.41698125]
beta : [0.09805757 0.11403222 0.32267886 0.27556015 0.22156274]


---
## 9 · Shuffling and sampling

These are essential for train/test splits, bootstrapping, and data augmentation.

In [16]:
rng = np.random.default_rng(42)

arr = np.arange(10)

# shuffle — modifies IN PLACE, returns None
rng.shuffle(arr)
print("shuffled:", arr)

# permutation — returns a NEW shuffled array, original unchanged
arr = np.arange(10)
perm = rng.permutation(arr)
print("permutation:", perm)
print("original:", arr)  # unchanged

# choice — sample k elements from an array
population = np.array([10, 20, 30, 40, 50])
print("with replacement:", rng.choice(population, size=3, replace=True))
print("without replacement:", rng.choice(population, size=3, replace=False))

# Practical: random train/test split indices
n = 100
idx = rng.permutation(n)
train_idx = idx[:80]
test_idx  = idx[80:]
print(f"\ntrain: {len(train_idx)}, test: {len(test_idx)}")


shuffled: [5 6 0 7 3 2 4 9 1 8]
permutation: [4 8 2 6 5 9 7 3 0 1]
original: [0 1 2 3 4 5 6 7 8 9]
with replacement: [40 30 50]
without replacement: [40 30 20]

train: 80, test: 20


---
## 10 · Histograms — understanding distributions

`np.histogram` counts how many values fall in each bin — the numerical
backbone behind any distribution plot.

In [ ]:
rng = np.random.default_rng(42)
data = rng.normal(0, 1, 1000)

counts, bin_edges = np.histogram(data, bins=10)
print("counts:", counts)
print("bin edges:", np.round(bin_edges, 2))
print("total:", counts.sum())   # should be 1000

# Normalised to probability density
counts_norm, _ = np.histogram(data, bins=10, density=True)
# Area under histogram ≈ 1.0
bin_width = bin_edges[1] - bin_edges[0]
print("area under density hist:", np.round(counts_norm.sum() * bin_width, 4))

counts    : [  1  13  44 131 218 296 179  87  25   6]
bin edges : [-3.65 -2.97 -2.28 -1.6  -0.92 -0.23  0.45  1.13  1.81  2.5   3.18]
total     : 1000
area under density hist: 1.0


---
## 11 · Summary cheatsheet

```python
# Descriptive statistics
np.mean(a)                # arithmetic mean
np.median(a)              # middle value
np.std(a, ddof=1)         # sample std  (ddof=1 for samples!)
np.var(a, ddof=1)         # sample variance
np.percentile(a, 75)      # 75th percentile
np.quantile(a, 0.75)      # same but fraction-based
np.corrcoef(x, y)         # Pearson r correlation matrix
np.cov(x, y)              # covariance matrix
np.histogram(a, bins=10)  # counts + bin edges

# Modern random API — ALWAYS use this
rng = np.random.default_rng(seed=42)

rng.random(n)                         # uniform [0,1)
rng.uniform(low, high, size)          # uniform [low, high)
rng.integers(low, high, size)         # integer [low, high)
rng.normal(loc, scale, size)          # Gaussian
rng.binomial(n, p, size)              # binomial
rng.poisson(lam, size)                # Poisson
rng.exponential(scale, size)          # exponential

rng.shuffle(arr)                      # in-place shuffle
rng.permutation(arr)                  # new shuffled copy
rng.choice(arr, size, replace=False)  # random sample
```

---
